In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("loan.csv")

In [3]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [9]:
df.dtypes

Loan_ID               object
Gender                object
Married               object
Dependents            object
Education             object
Self_Employed         object
ApplicantIncome        int64
CoapplicantIncome    float64
LoanAmount           float64
Loan_Amount_Term     float64
Credit_History       float64
Property_Area         object
Loan_Status           object
dtype: object

In [13]:
df_object = df.select_dtypes(include=['object']).columns

In [14]:
df_object

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'Property_Area', 'Loan_Status'],
      dtype='object')

In [31]:
df["Gender"].value_counts().columns

AttributeError: 'Series' object has no attribute 'columns'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.ensemble import RandomForestClassifier

# 1. Setup folders and naming
os.makedirs('images', exist_ok=True)
filename = 'tomato.csv'
food_name = 'TOMATO'

print(f"Generating {food_name} Graphs with AI Importance...")

# 2. Load and Pre-process Data
df = pd.read_csv(filename).rename(columns={
    'field1': 'Methane', 'field2': 'Temperature', 
    'field3': 'Humidity', 'field4': 'Moisture', 'field5': 'Ammonia', 
    'created_at': 'Timestamp'
}).dropna()

df['Timestamp'] = pd.to_datetime(df['Timestamp'])
start_time = df['Timestamp'].min()
df['Hours'] = (df['Timestamp'] - start_time).dt.total_seconds() / 3600

# 3. Calculate Feature Importance (40-20-40 Split)
# We define the Fresh (first 40%) and Rotten (last 40%) zones
split_1 = int(len(df) * 0.40)
split_2 = int(len(df) * 0.60)

train_df = pd.concat([
    df.iloc[:split_1].assign(Target=0), 
    df.iloc[split_2:].assign(Target=1)
])

X = train_df[['Methane', 'Ammonia', 'Moisture', 'Temperature', 'Humidity']]
y = train_df['Target']

# Train local Random Forest
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X, y)

# Get and sort importances
importances = clf.feature_importances_
imp_dict = {feat: round(imp * 100, 2) for feat, imp in zip(X.columns, importances)}
imp_dict = dict(sorted(imp_dict.items(), key=lambda item: item[1], reverse=True))

# 4. Create the 5-Panel Graph
fig, axes = plt.subplots(5, 1, figsize=(12, 15), sharex=True)
fig.suptitle(f'{food_name} Spoilage Profile (Detailed Analytics)', fontsize=16, fontweight='bold', y=0.98)

# Mapping sensors to specific colors for the PPT
sensors = [
    ('Moisture', 'blue', 'Raw Value'),
    ('Ammonia', 'purple', 'PPM'),
    ('Methane', 'red', 'PPM'),
    ('Humidity', 'teal', '%'),
    ('Temperature', 'orange', '°C')
]

for i, (name, col, unit) in enumerate(sensors):
    axes[i].plot(df['Hours'], df[name], color=col, linewidth=2)
    axes[i].set_ylabel(f'{name}\n({unit})', fontweight='bold')
    axes[i].grid(True, alpha=0.3)

axes[4].set_xlabel('Time Elapsed (Hours)', fontsize=12, fontweight='bold')

# 5. Add the AI Legend Box
legend_text = f"{food_name} IMPACT DATA\n" + "-"*22 + "\n"
for k, v in imp_dict.items():
    legend_text += f"{k}: {v}%\n"

# Stylized box for the PPT
props = dict(boxstyle='round', facecolor='#fff4e6', alpha=0.9, edgecolor='#d9480f')
fig.text(0.84, 0.5, legend_text.strip(), fontsize=11, verticalalignment='center', 
         bbox=props, family='monospace')

# 6. Save the final result
plt.tight_layout(rect=[0, 0, 0.82, 0.96])
save_path = os.path.join('images', 'tomato_analytics_legend.png')
plt.savefig(save_path, dpi=200, bbox_inches='tight')

print(f"✅ Success! Saved to {save_path}")z